# WER debug: n-gram LM, no-TTA, GRU vs Ours on Val Day 1

Loads the n-gram decoder and both models (baseline GRU + ours/MDAN), runs **no-TTA** eval on **val day 1 only** (first val session), and prints WER for each. Uses same logic as `run_n_gram_wer_eval.py`.

- **Refs:** `transcription_tensor_to_str(batch["transcriptions"][0])` then `clean_transcription(ref_raw)`.
- **Forced day:** Val → last train day (4) for both models; test → last val day for both.

# START HERE. TTA on test no val init


In [1]:
import os
import sys

BASE = "/victoriapvc"
REPO = os.path.join(BASE, "repos", "brain2text-t15")
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(BASE, "pip_packages"))

os.environ.setdefault("LM_DECODER_DIR", "/victoriapvc/data/willett/lm/languageModel")
os.environ.setdefault("BRAIN2TEXT_LM_DIR", "/victoriapvc/data/willett/lm/languageModel")

from pathlib import Path
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0


In [2]:
import numpy as np
import torch
from tta_utils import clean_transcription, get_phonemes
from lm_utils import _cer_and_wer
from model_training.data_augmentations import gauss_smooth




def freeze_day_affine_only(model):
	"""
	Freeze all params; unfreeze ONLY day-specific affine params (all days).
	Expected names: day_weights.<k>, day_biases.<k>
	"""
	for p in model.parameters():
		p.requires_grad = False

	for name, p in model.named_parameters():
		if name.startswith("day_weights.") or name.startswith("day_biases."):
			p.requires_grad = True

	trainable = [p for p in model.parameters() if p.requires_grad]
	if len(trainable) == 0:
		raise RuntimeError("No day_weights/day_biases params found. Check parameter names.")
	return trainable


def lm_decode_one(logits_1tC, ev):
	"""
	logits_1tC: [1, T, C]
	Returns a cleaned decoded string (B=1).
	"""
	T = int(logits_1tC.shape[1])
	decoded, _, _ = ev.get_lm_outputs(logits_1tC, n_frames=T)
	if isinstance(decoded, (list, tuple)):
		decoded = decoded[0]
	return clean_transcription(decoded)


def build_tta_optimizer_for_day_affine(
	model,
	*,
	lr: float,
	weight_decay: float,
	betas=(0.9, 0.999),
	eps: float = 0.1,
):
	"""
	Freezes model to day-affine-only and builds AdamW over those params.
	Returns: (trainable_params, optimizer)
	"""
	trainable_params = freeze_day_affine_only(model)
	optimizer = torch.optim.AdamW(
		trainable_params,
		lr=lr,
		weight_decay=weight_decay,
		betas=betas,
		eps=eps,
	)
	return trainable_params, optimizer


def apply_random_cut_and_get_logit_lens(
	X: torch.Tensor,  # [B, T, D]
	*,
	random_cut: int,
	model,
	device,
):
	"""
	Applies random_cut exactly like the trainer (cut from the left),
	then computes adjusted_lens = ((n_time_steps - patch_size)/patch_stride + 1).

	Returns:
	  X_cut: [B, T_cut, D]
	  logit_lens: [B] int32  (CTC input lengths)
	"""
	assert X.ndim == 3, f"Expected X [B,T,D], got {tuple(X.shape)}"

	# --- apply cut (same as trainer) ---
	if random_cut > 0:
		cut = int(np.random.randint(0, random_cut))
		if cut > 0:
			X = X[:, cut:, :]
	else:
		cut = 0

	# --- compute adjusted lens like trainer ---
	patch_size = getattr(model, "patch_size", None)
	patch_stride = getattr(model, "patch_stride", None)
	if patch_size is None or patch_stride is None:
		raise AttributeError(
			"Model missing patch_size/patch_stride attributes needed for adjusted_lens. "
			"Either add model.patch_size/model.patch_stride or pass them explicitly."
		)

	T = int(X.shape[1])
	n_time_steps = torch.full((X.shape[0],), T, device=device, dtype=torch.int32)  # [B]
	logit_lens = ((n_time_steps - patch_size) / patch_stride + 1).to(torch.int32)  # [B]

	return X, logit_lens


# ----------------------------
# Core streaming TTA stepper
# ----------------------------
def run_gru_tta_streaming(
	*,
	model,
	forced_day: int,
	loader,
	tta_args: dict,
	device,
	ev,
	optimizer,
	trainable_params,
	debug_first_n: int = 10,
	score_on: str = "pre",  # "pre" (streaming-compatible) or "post"
):
	"""
	Run streaming TTA over a loader (B=1).
	Uses optimizer passed in (no reset).
	Returns WER (float in [0,1]).
	"""
	assert score_on in {"pre", "post"}

	n_augs = int(tta_args["repeats"][0])
	adapt_steps = int(tta_args["adaptation_steps"])
	use_aug = bool(tta_args.get("WN+BS", False))

	# Aug params (match your trainer names as closely as possible)
	white_noise_std = float(tta_args.get("white_noise_std", tta_args.get("white_noise", 1.0)))
	constant_offset_std = float(tta_args.get("constant_offset_std", tta_args.get("baseline_shift", 0.2)))
	static_gain_std = float(tta_args.get("static_gain_std", 0.0))
	random_walk_std = float(tta_args.get("random_walk_std", 0.0))
	random_walk_axis = int(tta_args.get("random_walk_axis", -1))
	random_cut = int(tta_args.get("random_cut", 0))
	smooth_data = bool(tta_args.get("smooth_data", False))
	smooth_kernel_std = float(tta_args.get("smooth_kernel_std", 2.0))
	smooth_kernel_size = int(tta_args.get("smooth_kernel_size", 100))

	forced_day_t = torch.tensor([int(forced_day)], device=device, dtype=torch.long)

	decoded_list, ref_list = [], []

	for batch_idx, batch in enumerate(loader):
		features = batch["input_features"].to(device)  # [1, T, D]
		B = int(features.shape[0])
		assert B == 1, f"Expected B=1 streaming loader, got B={B}"

		day_idx = forced_day_t  # [1]

		# NOTE: transcription_tensor_to_str must exist in your namespace.
		ref_raw = transcription_tensor_to_str(batch["transcriptions"][0])
		ref = clean_transcription(ref_raw)

		# ---- PRE decode on CURRENT model ----
		model.eval()
		with torch.no_grad():
			logits_pre = model(features, day_idx)  # [1, T', C]
			decoded_pre = lm_decode_one(logits_pre, ev)

		if batch_idx < debug_first_n:
			print(f"\n[tta] trial {batch_idx}")
			print("  REF :", ref)
			print("  PRE :", decoded_pre)

		# Pseudo-label for this trial
		y_pseudo, y_len_pseudo = get_phonemes(decoded_pre)  # y_pseudo: [L]
		if not torch.is_tensor(y_pseudo):
			y_pseudo = torch.tensor(y_pseudo, dtype=torch.long)
		y_pseudo = y_pseudo.to(device)
		y_len_pseudo = torch.tensor([int(y_len_pseudo)], device=device, dtype=torch.int32)  # [1]

		# ---- TTA update (after decode) ----
		if adapt_steps > 0:
			# repeat trial n_augs times
			if n_augs > 1:
				X = features.repeat(n_augs, 1, 1)                # [n_augs, T, D]
				day_rep = day_idx.repeat(n_augs)                 # [n_augs]
				y_rep = y_pseudo.unsqueeze(0).repeat(n_augs, 1)  # [n_augs, L]
				ylen_rep = y_len_pseudo.repeat(n_augs)           # [n_augs]
			else:
				X = features
				day_rep = day_idx
				y_rep = y_pseudo.unsqueeze(0)                    # [1, L]
				ylen_rep = y_len_pseudo                          # [1]

			# Apply augmentations (trainer order)
			# (If you truly always use n_augs>1, leaving the condition is fine.
			#  If you want exact parity with trainer, use "if use_aug:" instead.)
			if use_aug and n_augs > 1:
				BB, Tt, Cc = X.shape
				if static_gain_std > 0:
					warp_mat = torch.eye(Cc, device=device).unsqueeze(0).repeat(BB, 1, 1)
					warp_mat = warp_mat + torch.randn_like(warp_mat) * static_gain_std
					X = torch.matmul(X, warp_mat)
				if white_noise_std > 0:
					X = X + torch.randn_like(X) * white_noise_std
				if constant_offset_std > 0:
					X = X + torch.randn((BB, 1, Cc), device=device) * constant_offset_std
				if random_walk_std > 0:
					X = X + torch.cumsum(torch.randn_like(X) * random_walk_std, dim=random_walk_axis)
				# random_cut handled below (so lengths match)
				if smooth_data:
					X = gauss_smooth(
						X,
						device=device,
						smooth_kernel_std=smooth_kernel_std,
						smooth_kernel_size=smooth_kernel_size,
					)

			# Random cut + correct CTC lengths (matches trainer behavior)
			X, logit_lens = apply_random_cut_and_get_logit_lens(
				X,
				random_cut=random_cut,
				model=model,
				device=device,
			)

			model.train()
			for _ in range(adapt_steps):
				logits = model(X, day_rep)  # [B, T', C]

				# IMPORTANT: assumes ev._ctc_loss_exact expects lengths in "post-patch" units,
				# like your trainer's adjusted_lens. If it expects logit lengths directly,
				# switch to logit_lens = torch.full((logits.shape[0],), logits.shape[1], ...).
				loss = ev._ctc_loss_exact(logits, y_rep, logit_lens, ylen_rep)

				optimizer.zero_grad(set_to_none=True)
				loss.backward()
				torch.nn.utils.clip_grad_norm_(trainable_params, 0.5)
				optimizer.step()

		# ---- optional POST decode ----
		model.eval()
		with torch.no_grad():
			logits_post = model(features, day_idx)
			decoded_post = lm_decode_one(logits_post, ev)

		if batch_idx < debug_first_n:
			print("  POST:", decoded_post)

		decoded_list.append(decoded_pre if score_on == "pre" else decoded_post)
		ref_list.append(ref)

	_, wer = _cer_and_wer(decoded_list, ref_list, outputType="speech", returnCI=False)
	return wer

In [3]:
# Import eval helpers from the main script (same LM, models, loader, WER)
import run_n_gram_wer_eval as ev

cfg = ev.split_config("split1")
baseline_dir = Path(cfg["baseline_dir"] + "/checkpoints")
ours_dir = Path(cfg["ours_dir"])
val_sessions = cfg["val_sess"]
test_sessions = cfg["test_sess"]
last_day_idx = cfg["last_day_idx"]  # last train day index (for GRU forced_day)
# Val day 1 = first val session only
sessions_eval = val_sessions
sessions_test = test_sessions[:4]
last_val_day_idx = ev.SESSIONS_26.index(val_sessions[0])  # day index for "ours" model

print(f"Baseline dir: {baseline_dir}")
print(f"Ours dir:     {ours_dir}")
print(f"Val day  session: {sessions_eval}")
print(f"Test day  session: {sessions_test}")
print(f"last_day_idx={last_day_idx} last_val_day_idx={last_val_day_idx}")

Baseline dir: /victoriapvc/data/outputs/baseline_split1/checkpoints
Ours dir:     /victoriapvc/data/outputs/20260218_s011_rep2_lam0.7_lrm0.8_wup5_hid256_dcp0_dd0_alpalt_alf24
Val day  session: ['t15.2023.08.27', 't15.2023.09.01', 't15.2023.09.03', 't15.2023.09.24', 't15.2023.09.29', 't15.2023.10.01', 't15.2023.10.06', 't15.2023.10.08', 't15.2023.10.13', 't15.2023.10.15', 't15.2023.10.20', 't15.2023.10.22', 't15.2023.11.03']
Test day  session: ['t15.2023.11.04', 't15.2023.11.17', 't15.2023.11.19', 't15.2023.11.26']
last_day_idx=4 last_val_day_idx=5


In [6]:
# Load args and build val + test loaders (all val sessions, all test sessions)
args_base = ev._load_yaml(str(baseline_dir / "args.yaml"))
args_ours = ev._load_yaml(str(ours_dir / "args.yaml"))
args_base.setdefault("dataset", {})
args_ours.setdefault("dataset", {})
args_base["dataset"]["sessions"] = ev.SESSIONS_26
args_ours["dataset"]["sessions"] = ev.SESSIONS_26

In [7]:
import numpy as np
def transcription_tensor_to_str(t):
	# t: torch.Tensor of ints (often uint8 / int) representing bytes or chars
	a = t.detach().cpu().numpy()

	# common case: uint8 bytes with trailing zeros
	if a.dtype != np.uint8:
		a = a.astype(np.uint8)

	# trim at first 0 (null terminator), if present
	if (a == 0).any():
		a = a[: int(np.where(a == 0)[0][0])]

	return bytes(a.tolist()).decode("utf-8", errors="ignore")

In [8]:
n_val_days = len(sessions_eval)
print(f"n_val_days={n_val_days}")
print(f"sessions_eval={sessions_eval}")
loaders_per_val_day = [
	ev._make_eval_loader_from_manifest(args_ours, [sessions_eval[d]], split_name="val")
	for d in range(n_val_days)
]
print(len(loaders_per_val_day))

n_val_days=13
sessions_eval=['t15.2023.08.27', 't15.2023.09.01', 't15.2023.09.03', 't15.2023.09.24', 't15.2023.09.29', 't15.2023.10.01', 't15.2023.10.06', 't15.2023.10.08', 't15.2023.10.13', 't15.2023.10.15', 't15.2023.10.20', 't15.2023.10.22', 't15.2023.11.03']
13


In [9]:
n_test_days = len(sessions_test)
print(f"n_test_days={n_test_days}")
print(f"sessions_test={sessions_test}")
loaders_per_test_day = [
	ev._make_eval_loader_from_manifest(args_ours, [sessions_test[d]], split_name="test")
	for d in range(n_test_days)
]
print(len(loaders_per_test_day))

n_test_days=4
sessions_test=['t15.2023.11.04', 't15.2023.11.17', 't15.2023.11.19', 't15.2023.11.26']
4


In [10]:
tta_args = {
	"repeats": [64],
	"adaptation_steps": 1,
	"WN+BS": True,
	"white_noise": 1.0,
	"baseline_shift": 0.2,
	"l2_decay": 0.0,
	'random_walk_std': 0.0,
	'random_walk_axis': -1,
	'static_gain_std': 0.0,
	'random_cut': 3,
	'smooth_data': True,
	'smooth_kernel_std': 2,
	'smooth_kernel_size': 100,
}
gru_tta_lr = 0.005
ours_tta_lr = 0.0005
last_val_idx = 4

# val day TTA baseline

In [11]:
from tta_utils import clean_transcription
from lm_utils import _cer_and_wer

In [12]:
last_val_idx

4

In [14]:
import json
import random

for seed in [1, 2, 3, 4, 5]:
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)  
	model_gru = ev._build_gru_from_args(args_base, device)
	ckpt_gru = ev._resolve_best_checkpoint(baseline_dir)
	ev._load_ckpt_state(model_gru, str(ckpt_gru), device, fill_missing_day_from_idx=last_val_idx)

	# -----------------------------
	# 2) Freeze day-affine only
	# -----------------------------
	for p in model_gru.parameters():
		p.requires_grad = False

	for name, p in model_gru.named_parameters():
		if name.startswith("day_weights.") or name.startswith("day_biases."):
			p.requires_grad = True

	trainable_params = [p for p in model_gru.parameters() if p.requires_grad]
	assert len(trainable_params) > 0, "No day affine params found."

	# -----------------------------
	# 3) Build optimizer ONCE
	#    (momentum carries across days)
	# -----------------------------
	wd = float(tta_args.get("l2_decay", 0.0))

	optimizer = torch.optim.AdamW(
		trainable_params,
		lr=gru_tta_lr,
		weight_decay=wd,
		betas=(0.9, 0.999),
		eps=0.1,
	)

	# -----------------------------
	# 4) Loop over val days
	# -----------------------------
	output_dir = "/victoriapvc/results/t15-final/split1"
	os.makedirs(output_dir, exist_ok=True)

	for val_day_idx in range(n_val_days):

		loader_day = loaders_per_val_day[val_day_idx]

		wer = run_gru_tta_streaming(
			model=model_gru,               # SAME model object
			forced_day=last_val_idx,
			loader=loader_day,
			tta_args=tta_args,
			device=device,
			ev=ev,
			optimizer=optimizer,           # SAME optimizer (no reset)
			trainable_params=trainable_params,
			debug_first_n=0,
			score_on="pre",                # streaming-compatible
		)

		print(f"val day {val_day_idx} WER: {wer:.2%}")

		out_path = os.path.join(
			output_dir,
			f"gru_wer_val_day{val_day_idx}_lr{gru_tta_lr}_seed{seed}.json",
		)

		with open(out_path, "w") as f:
			json.dump({"val_day_index": val_day_idx, "wer": float(wer)}, f, indent=2)
	# save the val day model ckpt
	torch.save(model_gru.state_dict(), os.path.join(output_dir, f"gru_val_day{val_day_idx}_lr{gru_tta_lr}_seed{seed}_ckpt.pth"))


[LM] RESCORE requested but G_no_prune.fst is missing; disabling rescoring for stability.


I0224 07:54:12.584064 18980 brain_speech_decoder.h:52] Reading fst /victoriapvc/data/willett/lm/languageModel/TLG.fst
I0224 08:02:41.959821 18980 brain_speech_decoder.h:81] Reading symbol table /victoriapvc/data/willett/lm/languageModel/words.txt
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


val day 0 WER: 20.90%
val day 1 WER: 37.84%
val day 2 WER: 43.41%
val day 3 WER: 81.33%
val day 4 WER: 99.69%
val day 5 WER: 99.82%
val day 6 WER: 100.00%
val day 7 WER: 99.95%
val day 8 WER: 100.00%
val day 9 WER: 99.94%
val day 10 WER: 100.00%
val day 11 WER: 100.00%
val day 12 WER: 100.00%
val day 0 WER: 21.18%
val day 1 WER: 38.12%
val day 2 WER: 43.90%
val day 3 WER: 86.15%
val day 4 WER: 99.69%
val day 5 WER: 99.82%
val day 6 WER: 100.00%
val day 7 WER: 99.79%
val day 8 WER: 100.00%
val day 9 WER: 100.00%
val day 10 WER: 100.00%
val day 11 WER: 100.00%
val day 12 WER: 100.00%
val day 0 WER: 21.27%
val day 1 WER: 38.64%
val day 2 WER: 44.35%
val day 3 WER: 84.84%
val day 4 WER: 99.92%
val day 5 WER: 99.88%
val day 6 WER: 100.00%
val day 7 WER: 99.90%
val day 8 WER: 100.00%
val day 9 WER: 100.00%
val day 10 WER: 100.00%
val day 11 WER: 100.00%
val day 12 WER: 100.00%
val day 0 WER: 20.99%
val day 1 WER: 37.93%
val day 2 WER: 43.68%
val day 3 WER: 82.82%
val day 4 WER: 99.84%
val da

## val day TTA for ours

In [ ]:
import json
import random

for seed in [1, 2, 3, 4, 5]:
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)  
	model_ours = ev._build_gru_mdan_from_args(args_ours, device)
	ckpt_ours = ev._resolve_best_checkpoint(ours_dir)
	ev._load_ckpt_state(model_ours, str(ckpt_ours), device, fill_missing_day_from_idx=last_val_idx)

	# -----------------------------
	# 2) Freeze day-affine only
	# -----------------------------
	for p in model_ours.parameters():
		p.requires_grad = False

	for name, p in model_ours.named_parameters():
		if name.startswith("day_weights.") or name.startswith("day_biases."):
			p.requires_grad = True

	trainable_params = [p for p in model_ours.parameters() if p.requires_grad]
	assert len(trainable_params) > 0, "No day affine params found."

	# -----------------------------
	# 3) Build optimizer ONCE
	#    (momentum carries across days)
	# -----------------------------
	wd = float(tta_args.get("l2_decay", 0.0))

	optimizer = torch.optim.AdamW(
		trainable_params,
		lr=ours_tta_lr,
		weight_decay=wd,
		betas=(0.9, 0.999),
		eps=0.1,
	)

	# -----------------------------
	# 4) Loop over val days
	# -----------------------------
	output_dir = "/victoriapvc/results/t15-final/split1"
	os.makedirs(output_dir, exist_ok=True)

	for val_day_idx in range(n_val_days):

		loader_day = loaders_per_val_day[val_day_idx]

		wer = run_gru_tta_streaming(
			model=model_ours,               # SAME model object
			forced_day=last_val_idx,
			loader=loader_day,
			tta_args=tta_args,
			device=device,
			ev=ev,
			optimizer=optimizer,           # SAME optimizer (no reset)
			trainable_params=trainable_params,
			debug_first_n=0,
			score_on="pre",                # streaming-compatible
		)

		print(f"val day {val_day_idx} WER: {wer:.2%}")

		out_path = os.path.join(
			output_dir,
			f"ours_wer_val_day{val_day_idx}_lr{ours_tta_lr}_seed{seed}.json",
		)

		with open(out_path, "w") as f:
			json.dump({"val_day_index": val_day_idx, "wer": float(wer)}, f, indent=2)
	# save the val day model ckpt
	torch.save(model_ours.state_dict(), os.path.join(output_dir, f"ours_val_day{val_day_idx}_lr{ours_tta_lr}_seed{seed}_ckpt.pth"))


# Test Day TTA

In [14]:
import json
import random

for seed in [1, 2, 3, 4, 5]:
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)  
	model_gru = ev._build_gru_from_args(args_base, device)
	ckpt_gru = ev._resolve_best_checkpoint(baseline_dir)
	ev._load_ckpt_state(model_gru, str(ckpt_gru), device, fill_missing_day_from_idx=last_val_idx)

	# -----------------------------
	# 2) Freeze day-affine only
	# -----------------------------
	for p in model_gru.parameters():
		p.requires_grad = False

	for name, p in model_gru.named_parameters():
		if name.startswith("day_weights.") or name.startswith("day_biases."):
			p.requires_grad = True

	trainable_params = [p for p in model_gru.parameters() if p.requires_grad]
	assert len(trainable_params) > 0, "No day affine params found."

	# -----------------------------
	# 3) Build optimizer ONCE
	#    (momentum carries across days)
	# -----------------------------
	wd = float(tta_args.get("l2_decay", 0.0))

	optimizer = torch.optim.AdamW(
		trainable_params,
		lr=gru_tta_lr,
		weight_decay=wd,
		betas=(0.9, 0.999),
		eps=0.1,
	)

	# -----------------------------
	# 4) Loop over test days
	# -----------------------------
	output_dir = "/victoriapvc/results/t15-final/split1"
	os.makedirs(output_dir, exist_ok=True)

	for test_day_idx in range(n_test_days):

		loader_day = loaders_per_test_day[test_day_idx]

		wer = run_gru_tta_streaming(
			model=model_gru,               # SAME model object
			forced_day=last_val_idx,
			loader=loader_day,
			tta_args=tta_args,
			device=device,
			ev=ev,
			optimizer=optimizer,           # SAME optimizer (no reset)
			trainable_params=trainable_params,
			debug_first_n=0,
			score_on="pre",                # streaming-compatible
		)

		print(f"test day {test_day_idx} WER: {wer:.2%}")

		out_path = os.path.join(
			output_dir,
			f"gru_wer_test_day_fixed_len_rancut_{test_day_idx}_lr{gru_tta_lr}_seed{seed}.json",
		)

		with open(out_path, "w") as f:
			json.dump({"test_day_index": test_day_idx, "wer": float(wer)}, f, indent=2)

test day 0 WER: 52.57%
test day 1 WER: 90.58%
test day 2 WER: 99.19%
test day 3 WER: 100.00%
test day 0 WER: 50.00%
test day 1 WER: 85.43%
test day 2 WER: 99.19%
test day 3 WER: 100.00%
test day 0 WER: 51.10%
test day 1 WER: 86.31%
test day 2 WER: 98.98%
test day 3 WER: 100.00%
test day 0 WER: 51.65%
test day 1 WER: 86.56%
test day 2 WER: 99.19%
test day 3 WER: 99.94%
test day 0 WER: 51.10%
test day 1 WER: 84.67%
test day 2 WER: 98.98%
test day 3 WER: 99.94%


# test TTA ours

In [13]:
import json
import random
for seed in [1, 2, 3, 4, 5]:
	random.seed(seed)
	np.random.seed(seed)
	torch.manual_seed(seed)
	torch.cuda.manual_seed_all(seed)   
	model_ours = ev._build_gru_mdan_from_args(args_ours, device)
	ckpt_ours = ev._resolve_best_checkpoint(ours_dir)
	ev._load_ckpt_state(model_ours, str(ckpt_ours), device, fill_missing_day_from_idx=last_val_idx)

	# -----------------------------
	# 2) Freeze day-affine only
	# -----------------------------
	for p in model_ours.parameters():
		p.requires_grad = False

	for name, p in model_ours.named_parameters():
		if name.startswith("day_weights.") or name.startswith("day_biases."):
			p.requires_grad = True

	trainable_params = [p for p in model_ours.parameters() if p.requires_grad]
	assert len(trainable_params) > 0, "No day affine params found."

	# -----------------------------
	# 3) Build optimizer ONCE
	#    (momentum carries across days)
	# -----------------------------
	wd = float(tta_args.get("l2_decay", 0.0))

	optimizer = torch.optim.AdamW(
		trainable_params,
		lr=ours_tta_lr,
		weight_decay=wd,
		betas=(0.9, 0.999),
		eps=0.1,
	)

	# -----------------------------
	# 4) Loop over test days
	# -----------------------------
	output_dir = "/victoriapvc/results/t15-final/split1"
	os.makedirs(output_dir, exist_ok=True)

	for test_day_idx in range(n_test_days):

		loader_day = loaders_per_test_day[test_day_idx]

		wer = run_gru_tta_streaming(
			model=model_ours,               # SAME model object
			forced_day=last_val_idx,
			loader=loader_day,
			tta_args=tta_args,
			device=device,
			ev=ev,
			optimizer=optimizer,           # SAME optimizer (no reset)
			trainable_params=trainable_params,
			debug_first_n=0,
			score_on="pre",                # streaming-compatible
		)

		print(f"test day {test_day_idx} ALIGN WER: {wer:.2%}")

		out_path = os.path.join(
			output_dir,
			f"align_wer_test_day_fixed_len_rancut_{test_day_idx}_lr{ours_tta_lr}_seed{seed}.json",
		)

		with open(out_path, "w") as f:
			json.dump({"test_day_index": test_day_idx, "wer": float(wer)}, f, indent=2)

[LM] RESCORE requested but G_no_prune.fst is missing; disabling rescoring for stability.


I0225 03:11:42.733358 135944 brain_speech_decoder.h:52] Reading fst /victoriapvc/data/willett/lm/languageModel/TLG.fst
I0225 03:19:44.468748 135944 brain_speech_decoder.h:81] Reading symbol table /victoriapvc/data/willett/lm/languageModel/words.txt
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


test day 0 ALIGN WER: 45.40%
test day 1 ALIGN WER: 65.70%
test day 2 ALIGN WER: 71.34%
test day 3 ALIGN WER: 86.68%
test day 0 ALIGN WER: 45.40%
test day 1 ALIGN WER: 65.83%
test day 2 ALIGN WER: 72.56%
test day 3 ALIGN WER: 86.23%
test day 0 ALIGN WER: 45.40%
test day 1 ALIGN WER: 65.70%
test day 2 ALIGN WER: 70.93%
test day 3 ALIGN WER: 86.49%
test day 0 ALIGN WER: 45.40%
test day 1 ALIGN WER: 65.70%
test day 2 ALIGN WER: 71.75%
test day 3 ALIGN WER: 86.74%
test day 0 ALIGN WER: 45.40%
test day 1 ALIGN WER: 65.33%
test day 2 ALIGN WER: 72.36%
test day 3 ALIGN WER: 86.74%
